<a href="https://colab.research.google.com/github/goumze/Simplilearn_Agentic_AI/blob/feature%2Fcollab/CS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Project Agenda: Build Banking Customer Support AI Agent with AI Agent Framework

# Description: You are developing an intelligent banking customer support system leveraging a multi-
# agent architecture.This AI-powered assistant is designed to address customer feedback and
# inquiries related to previously raised service tickets. The system is responsible for three
# primary operations:
# 1. Categorizing customer input as positive feedback, negative feedback, or service query
# 2. Delivering contextually relevant and empathetic responses based on the input
#     classification
# 3. Generating or retrieving ticket-related information from database as necessary.

# Any modern agent-based framework discussed during the training can be utilized to build
# This solution.


# Agent Responsibilities
# Agent 1: Classifier Agent
# Input: Unstructured customer text input (e.g., “Thanks for resolving my credit card issue!”,
#                      “My loan issue is still pending”, “Could you provide the update on ticket #450987?”)
# Task:
# - Analyze and classify input into one of the following categories:
# - Positive Feedback
# - Negative Feedback
# - Query
# Agent 2: Feedback Handler Agent .
# Triggered when the input is classified as Feedback.
# - For Positive Feedback:
# - Return a personalized thank-you message generated by a language model (e.g.,                  “Thank you for your kind words, [CustomerName]! We’re delighted to assist you.”)
# - For Negative Feedback:
# - Generate a new unique ticket number (e.g., a 6-digit code)
# - Insert the new ticket into a centralized database (table: `support_tickets`) with status
#    Unresolved;
# - Return a polite, empathetic response including the new ticket number (e.g., “We  regret the inconvenience. A new ticket #[TicketNumber] has been generated and our team will
#             follow up shortly.”)

# Agent 3: Query Handler Agent
# Triggered when the input is identified as a Query.

# Task:
# - Extract the ticket number from the customer input
# - Query the database (table: `support_tickets`) to retrieve the status of the associated ticket
# - Return a concise status update to the user (e.g., “Your ticket #[TicketNumber] is currently
# marked as: In Progress”)



I planned the design of my agentic workflow in below format -

**Classifier_agent:** to classify user input is a query or feedback ->


1.   **feedback_handler_agent:** If user input is feedback, then analyze the sentiment of the feedback (*SentimentAnalysisTool*)
2.   **query_handler_agent:** If user input is query, then look for query status (*query_status_tool*) if its an existing query, else create new ticket number (*ticket_generator_tool*). If the user input is simply a positive feedback, do not create ticket.


    



In [5]:
 !pip install crewai crewai_tools -q
 !pip install transformers -q
 !pip install langchain_huggingface -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.1/811.1 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [6]:
# creare tools
# this is for custom tools
from crewai.tools import BaseTool

In [7]:
# Load the .env file
from dotenv import load_dotenv
import os
load_dotenv()

False

In [8]:
from crewai import LLM
llm = LLM(model="gpt-3.5-turbo")

In [9]:
# tool 1: Response classifer whether its feedback or query
from pydantic import BaseModel, Field
from crewai.tools import BaseTool
# Assuming llm is defined earlier as shown in the original notebook

class ToolInput(BaseModel):
  user_input: str = Field(..., description="input")

class ResponseClassifierTool(BaseTool):
    name: str = "Response Classifier Tool"
    description: str = "Classify the user_input as Feedback or Query"
    # Corrected args_schema to use the ToolInput model
    args_schema: type = ToolInput

    def _run(self, user_input: str) -> str:
        # Construct a specific prompt for the LLM to guide its output
        classification_prompt = f"Classify the following user input as either 'query' or 'feedback'. Respond with only one word: '{user_input}'"

        # Call the LLM with the specific prompt
        raw_classification = llm.call(classification_prompt)

        # Process the raw output to ensure it's either 'query' or 'feedback'
        # This is a basic example; you might need more robust handling
        classified_output = raw_classification.strip().lower()

        if classified_output not in ['query', 'feedback']:
            # Handle cases where the LLM doesn't return the expected output
            print(f"Warning: LLM returned unexpected classification: {raw_classification}. Defaulting to 'query'.")
            classified_output = 'query' # Or some other default/error handling

        print(f"Classified user_input as: {classified_output}")
        return classified_output

In [10]:
# tool 2: Sentiment analyzer
from crewai.tools import BaseTool # Ensure BaseTool is imported if not already
from transformers import pipeline # Import pipeline from transformers

class SentimentAnalysisTool(BaseTool):
    name: str = "Sentiment Analysis Tool"
    description: str = "Determines the sentiment of the user's response and returns 'Positive' or 'Negative'."

    def _run(self, input: str) -> str:
        sentiment_analyzer = pipeline("sentiment-analysis")
        result = sentiment_analyzer(input)

        # label the sentiment
        sentiment = result[0]['label'].title()

        # Print for debugging (optional)
        print(f"Sentiment analysis result for '{input}': {sentiment}")

        # Return 'Positive' or 'Negative'
        return sentiment

In [11]:
# tool 3: Respond to user feedback appropriately

from pydantic import BaseModel, Field

class ToolInput(BaseModel):
  feedback: str = Field(..., description="feedback")

class GenerationTool(BaseTool):
  name: str = "Response Generation Tool"
  description: str = "Respond to user input appropriately. use this tool to acknowledge and respond to user feedback in a respective manner."
  args_schema: type = ToolInput

#override run here
  def _run(self, feedback):
    "use this tool to use llm to respond to user's feedback"
    return llm.call(feedback)

response_generation_tool = GenerationTool()


In [12]:
# tool 4: Generate 6 digit random number and append to file

from crewai.tools import BaseTool
from pydantic import BaseModel, Field
import random
import os
import re  # Import re for regular expressions

class TicketInput(BaseModel):
    user_input: str = Field(default="", description="Optional user input text")

class TicketGeneratorTool(BaseTool):
    name: str = "Ticket Generator Tool"
    description: str = "Generate a unique 6 digit random number and append to file tickets.txt with status as in-progress. If a valid existing ticket is already present in user input, do nothing."
    args_schema: type = TicketInput

    def _run(self, user_input: str = "") -> str:
        file_path = "tickets.txt"
        generated_ticket = None
        max_attempts = 3

        # Check for a 6-digit number in user input
        match = re.search(r"\b(\d{6})\b", user_input)
        user_provided_ticket = match.group(1) if match else None

        existing_tickets = {}
        # Read existing tickets if file exists
        if os.path.exists(file_path):
            try:
                with open(file_path, "r") as f:
                    for line in f:
                        parts = line.strip().split(": ")
                        if len(parts) == 2:
                            existing_tickets[parts[0]] = parts[1]
            except Exception as e:
                return f"Error reading tickets.txt: {e}"

        # If a user-provided ticket exists in the file, no new ticket is needed
        if user_provided_ticket and user_provided_ticket in existing_tickets:
            return f"Ticket number {user_provided_ticket} already exists. No new ticket generated."

        # Generate a unique new ticket
        for _ in range(max_attempts):
            ticket_number = str(random.randint(100000, 999999))
            if ticket_number not in existing_tickets:
                generated_ticket = ticket_number
                break

        if generated_ticket:
            try:
                prefix = "\n" if os.path.exists(file_path) and os.path.getsize(file_path) > 0 else ""
                with open(file_path, "a") as f:
                    f.write(f"{prefix}{generated_ticket}: in-progress")
                return f"Generated new ticket number: {generated_ticket} with status in-progress."
            except Exception as e:
                return f"Error writing to tickets.txt: {e}"

        return "Could not generate a unique ticket number after multiple attempts."

ticket_generator_tool = TicketGeneratorTool()

In [13]:
# tool 5: Respond to query status by looking into the ticket.txt

from pydantic import BaseModel, Field
from crewai.tools import BaseTool
import re

class QueryInput(BaseModel):
  user_input: str = Field(..., description="user_input")

class QueryStatusTool(BaseTool):
    name: str = "query_status_tool"
    description: str = 'use this method to recognize when a user is asking for a ticket status and to use this "query_status_tool" with the user_input as input '
    args_schema: type = QueryInput

    def _run(self, user_input: str) -> str:
      """
      Retrieves the status of a ticket from tickets.txt based on a 6-digit number in the user's input.
      """
      # Extract the 6-digit ticket number from the user_query
      match = re.search(r'\b(\d{6})\b', user_input)
      if not match:
          return "Could not find a 6-digit ticket number, please suppy correct ticket number."

      ticket_number = match.group(1) # Extract the 6-digit number from the user input
      ticket_status = None # Initialize to None for explicit checking if found

      try:
          with open('tickets.txt', 'r') as f:
              for line in f:
                  parts = line.strip().split(': ')
                  if len(parts) == 2 and parts[0] == ticket_number:
                      ticket_status = parts[1]
                      break # Stop searching once the ticket is found
      except FileNotFoundError:
          return "The tickets.txt file was not found."
      except Exception as e:
          return f"An error occurred while reading the tickets file: {e}"

      if ticket_status is None: # Check if ticket_status is still None
          return "Ticket Number not found, please supply correct ticket number."
      else:
          return f"The status for ticket number {ticket_number} is: {ticket_status}"

query_status_tool = QueryStatusTool()

In [14]:
# now the 5 tools are ready, lets build Agents
# First Agent is Classifier_agent

from crewai import Agent

# Define the Classifier Agent with its role, goal, and backstory
classifier_agent = Agent(
    role="Classifier",
    goal="Classify the user response as feedback or a query, and if it's feedback, analyze its sentiment as Positive or Negative.",
    backstory=(
        "You are an expert classifier and sentiment analyst. Your first step is always to classify the user input as either 'query' or 'feedback'. "
        "If the input is classified as 'feedback', you then use your sentiment analysis skills to determine if the feedback is 'Positive' or 'Negative'."
    ),
    tools=[ResponseClassifierTool(), SentimentAnalysisTool()],
    llm=llm,
    verbose=False,
    allow_delegation=False
)

In [15]:
# Second Agent is the Feedback Handler Agent

feedback_handler_agent = Agent(
    role = "feedback_handler",
    goal = "use response_generation_tool to respond to the user's response or question",
    backstory=(
        " you are expert call center assistant who helps by acknowledging user's message "
        " depending on user's message, return a thankyou message, appreciating their positive feedback or express apologies along with assurance to serve them better in future "
    ),
    llm=llm,
    verbose=False,
    allow_delegation=False,
    tools=[response_generation_tool]
)

In [16]:
# Third Agent is the Query Handler Agent

query_handler_agent = Agent(
    role = "query_handler",
    goal = "Analyze user queries. If a 6-digit ticket number is present, use the query_status_tool to find its status. If no ticket number is present, use the ticket_generator_tool to create a new ticket.",
    backstory=(
        "You are a diligent assistant specializing in handling user inquiries related to support tickets. "
        "You are adept at identifying ticket numbers within user messages and either retrieving their status "
        "or generating new tickets when necessary."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    tools=[query_status_tool, ticket_generator_tool]
)

In [17]:
# Build Task - Classify Repsonse b/w Feedback or Query, Classify sentiment between positive and negative

from crewai import Task
classifier_task = Task(
    description= (" analyze the user response words and sense of user_input: {input} "
                  " based on words present in user input. decide whether it is a query or a feedback "
                  " return a single word 'query', if user input contains a 6 digit ticket number and/or is a question "
                  " return a single word 'feedback', if user response contains any feedback positive or negative "
                  " If classified as feedback, also determine its sentiment (Positive or Negative)."
                  ),
    expected_output=(" based on user input, return a single word from the ['query', 'feedback'] and if it is feedback then return its sentiment (Positive or Negative). " ), # Clarified expected output
    agent=classifier_agent
)

In [18]:
from crewai import Task
feedback_handler_task = Task(
    description=(
        "Use this task only for feedback input. User input: {input}. "
        "If feedback is positive, return a short thank-you message. "
        "If feedback is negative, return an empathetic apology and mention that support will follow up. "
        "If the input is a query, return: 'No feedback handling required.'"
    ),
    expected_output=(
        "A context-aware feedback response, or 'No feedback handling required.' for query input."
    ),
    agent=feedback_handler_agent
)

In [19]:
from crewai import Task

query_handler_task = Task(
    description=(
        "Analyze the user input: {input}. "
        "Only handle this task when the input is a query. "
        "If a 6-digit ticket number is present, use query_status_tool to return the ticket status. "
        "If no 6-digit ticket number is present, use ticket_generator_tool with user_input={input} to create a new ticket and return a helpful message with the new ticket number. "
        "If the input is feedback, return: 'No query handling required.'"
    ),
    expected_output=(
        "Ticket status for an existing ticket, or a newly generated ticket response, or 'No query handling required.'"
    ),
    agent=query_handler_agent,
    context=[classifier_task]
)

In [20]:
# now that the tools/agents/tasks are ready, create a deterministic router for final execution
import re

def run_support_flow(user_input: str) -> str:
    """Route customer input to the correct operation as per project requirements."""
    classification = ResponseClassifierTool()._run(user_input)

    if classification == "query":
        if re.search(r"\b\d{6}\b", user_input):
            return query_status_tool._run(user_input)
        ticket_result = ticket_generator_tool._run(user_input)
        return f"We have created a new support ticket for your query. {ticket_result}"

    # Feedback flow: detect sentiment first
    sentiment = SentimentAnalysisTool()._run(user_input)
    if sentiment.lower() == "positive":
        prompt = (
            "Create a short, warm thank-you response for this customer feedback: "
            f"{user_input}"
        )
        try:
            return response_generation_tool._run(prompt)
        except Exception:
            return "Thank you for your kind feedback. We are delighted to assist you."

    # Negative feedback flow: create ticket and return empathetic response
    ticket_result = ticket_generator_tool._run(user_input)
    ticket_match = re.search(r"(\d{6})", ticket_result)
    if ticket_match:
        ticket_no = ticket_match.group(1)
        return (
            f"We are sorry for the inconvenience. A new ticket #{ticket_no} has been created "
            "and our support team will follow up shortly."
        )
    return (
        "We are sorry for the inconvenience. A ticket has been raised and our support team "
        "will follow up shortly."
    )

In [25]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [26]:
result = run_support_flow("I want to share that I appreciate the last call with the customer agent, she was really helpful")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Classified user_input as: feedback


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Sentiment analysis result for 'I want to share that I appreciate the last call with the customer agent, she was really helpful': Positive


In [27]:
print(result)

Thank you for your kind words! We are so glad to hear that our customer agent was able to assist you. We truly value your feedback and look forward to helping you in the future.


In [28]:
result_1 = run_support_flow("I want to report issue with my last purchase.")
print(result_1)

Classified user_input as: query
We have created a new support ticket for your query. Generated new ticket number: 225691 with status in-progress.


In [30]:
result_2 = run_support_flow("What is the status of my last query #123456?")
print(result_2)

Classified user_input as: query
Ticket Number not found, please supply correct ticket number.
